In [1]:
from pathlib import Path
import pandas as pd
import re

# ========= 改成你的路徑 =========
input_csv = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/hhcomp_tables/ethnicity_household_shares.csv")
output_tex = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/hhcomp_tables/ethnicity_household_shares.tex")
# ===============================

def latex_escape(text):
    if pd.isna(text):
        return ""
    text = str(text)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    pattern = re.compile("|".join(re.escape(k) for k in replacements))
    return pattern.sub(lambda m: replacements[m.group(0)], text)

df = pd.read_csv(input_csv)

cols = [
    "ethn_group",
    "p_singlehh",
    "p_singlekids",
    "p_nokids",
    "p_1to2kids",
    "p_3pkids",
    "n",
]
missing = [c for c in cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV 缺少欄位: {missing}")

df = df[cols].copy()

num_cols = ["p_singlehh", "p_singlekids", "p_nokids", "p_1to2kids", "p_3pkids", "n"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

ethnicity_order = [
    "British/English/Scottish/Welsh/Northern Irish",
    "Indian",
    "Pakistani",
    "Bangladeshi",
    "African",
    "Caribbean",
    "Any other white background",
]
df["ethn_group"] = pd.Categorical(
    df["ethn_group"],
    categories=ethnicity_order,
    ordered=True
)
df = df.sort_values("ethn_group").reset_index(drop=True)

df["ethn_group"] = df["ethn_group"].astype(str).map(latex_escape)

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{Household Composition by Ethnicity}")
lines.append(r"\label{tab:ethnicity_household}")
lines.append(r"\begin{threeparttable}")
lines.append(r"\footnotesize")
lines.append(r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.2cm}cccccc}")
lines.append(r"\toprule")
lines.append(r"& \multicolumn{2}{c}{Household structure} & \multicolumn{3}{c}{Dependent children} & \\")
lines.append(r"\cmidrule(lr){2-3}\cmidrule(lr){4-6}")
lines.append(r"Ethnicity & Single & Single with kids & No kids & 1--2 kids & 3+ kids & N \\")
lines.append(r"\midrule")

for _, row in df.iterrows():
    eth = row["ethn_group"]
    single = "" if pd.isna(row["p_singlehh"]) else f'{row["p_singlehh"]:.2f}'
    singlekids = "" if pd.isna(row["p_singlekids"]) else f'{row["p_singlekids"]:.2f}'
    nokids = "" if pd.isna(row["p_nokids"]) else f'{row["p_nokids"]:.2f}'
    kids12 = "" if pd.isna(row["p_1to2kids"]) else f'{row["p_1to2kids"]:.2f}'
    kids3p = "" if pd.isna(row["p_3pkids"]) else f'{row["p_3pkids"]:.2f}'
    n = "" if pd.isna(row["n"]) else f'{int(round(row["n"])):,}'
    lines.append(f"{eth} & {single} & {singlekids} & {nokids} & {kids12} & {kids3p} & {n} \\\\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular*}")
lines.append(r"\begin{tablenotes}[flushleft]")
lines.append(r"\footnotesize")
lines.append(
    r"\item Notes: This table reports weighted percentages of household composition by ethnicity. Single with kids refers to single-person households with dependent children. The children columns report the distribution of the number of dependent children at baseline. Percentages are weighted using the CA Covid survey weights, and \(N\) denotes the unweighted sample size."
)
lines.append(r"\end{tablenotes}")
lines.append(r"\end{threeparttable}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
output_tex.write_text(latex_table, encoding="utf-8")

print(f"LaTeX table saved to: {output_tex}")
print()
print(latex_table)

LaTeX table saved to: /Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/hhcomp_tables/ethnicity_household_shares.tex

\begin{table}[htbp]
\centering
\caption{Household Composition by Ethnicity}
\label{tab:ethnicity_household}
\begin{threeparttable}
\footnotesize
\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.2cm}cccccc}
\toprule
& \multicolumn{2}{c}{Household structure} & \multicolumn{3}{c}{Dependent children} & \\
\cmidrule(lr){2-3}\cmidrule(lr){4-6}
Ethnicity & Single & Single with kids & No kids & 1--2 kids & 3+ kids & N \\
\midrule
British/English/Scottish/Welsh/Northern Irish & 4.69 & 3.03 & 71.01 & 25.25 & 3.74 & 10,910 \\
Indian & 0.67 & 0.66 & 58.33 & 38.52 & 3.15 & 409 \\
Pakistani & 4.98 & 0.91 & 67.92 & 17.62 & 14.46 & 257 \\
Bangladeshi & 0.86 & 0.26 & 48.10 & 30.86 & 21.05 & 98 \\
African & 7.53 & 6.98 & 53.87 & 35.81 & 10.32 & 93 \\
Caribbean & 7.75 & 6.09 & 68.06 & 30.01 & 1.93 & 102 \\
Any other white background & 4.25 & 3.98 & 53.29 & 43.8